## Map-Reduce Recursive Summarization

Implements a map-reduce style recursive summarization experiment over Notion docs.

Steps:
1. Load Notion markdown pages into a dictionary.
2. Select `block_reference` only.
3. Chunk with `RecursiveChunker` at chunk size `10000`.
4. Summarize grouped chunks in parallel using `async_chat_wrapper` with round-robin model selection.
5. Repeat map-reduce rounds until final summary length is under `20000` characters.

In [1]:
from __future__ import annotations

import asyncio
from pathlib import Path
from typing import Any
from itertools import cycle

from chonkie import RecursiveChunker

from src.all_functionality import async_chat_wrapper

DEFAULT_MODELS = ["gemma4", "gemma12", "gemma27"]


def load_notion_pages(docs_dir: str = "./rag_exp/folders/top25") -> dict[str, str]:
    """Load markdown notion pages into a dictionary keyed by file stem."""
    base = Path(docs_dir)
    pages: dict[str, str] = {}
    for p in sorted(base.glob("**/*.md")):
        with p.open("r", encoding="utf-8") as fh:
            pages[p.stem] = fh.read()
    return pages


def select_pages(pages: dict[str, str], keys: list[str] | None = None) -> dict[str, str]:
    """Select only the requested page keys from loaded pages."""
    if not keys:
        return pages
    return {k: v for k, v in pages.items() if k in set(keys)}


def chunk_with_recursive_chunker(text: str, chunk_size: int = 10000) -> list[str]:
    """Chunk text with chonkie's RecursiveChunker and normalize to plain strings."""
    chunker = RecursiveChunker(chunk_size=chunk_size)
    chunks = chunker.chunk(text)
    return [getattr(c, "text", str(c)).strip() for c in chunks if getattr(c, "text", str(c)).strip()]


def build_extractive_summary_prompt(chunk_text: str) -> str:
    return f"""
You are a precise assistant summarizing technical documentation.

Rules:
- Be strictly extractive and grounded in the provided text.
- Prefer quoting exact field names, endpoints, parameters, enums, constraints, and literal values.
- Do not generalize or add background explanations.
- Do not infer beyond the text.
- Keep output compact but high-density with concrete details.

<extraction_focus>
1. [VERBS/ENDPOINTS]: Extract exact HTTP methods, URL paths, and required headers for ALL Page, Database, Block, and Comment endpoints. Do not limit to specific operations; capture the full CRUD suite for these four core objects.
2. [NOUNS/SCHEMAS]: Distill strict write-payload JSON schemas for all standard property types (Title, Date, Select, Multi-Select, Status, Relation, Checkbox, Number, Rich Text). 
3. [LOGIC/FILTERS]: Isolate precise JSON structures for compound filtering (AND/OR) and all standard property filters (equals, contains, on_or_before, is_empty), plus Sorting rules.
4. [CONTENT/BLOCKS]: Pull raw, minimal JSON creation payloads for the complete textual block ecosystem (Paragraph, To-Do, Heading_1/2/3, Bulleted/Numbered Lists, Toggle, Quote, Callout, and standard Rich Text). 
5. [OPERATIONAL CONSTRAINTS]: Extract a concise, bulleted list of strict architectural limits, pagination maximums, and default boolean states.
6. [EXCLUSION ZONE]: You MUST IGNORE and EXCLUDE all documentation regarding media (images, audio, video, PDFs, files), user management, workspaces, authentication flows, OAuth, and SIEM compliance.
</extraction_focus>

Text to summarize:
<document_chunk>
{chunk_text}
</document_chunk>
""".strip()


async def summarize_single_chunk(
    chunk_text: str,
    model_name: str,
    temperature: float = 0.1,
    extra_instruction: str | None = None,
) -> str:
    """Summarize one chunk with the selected model."""
    prompt = build_extractive_summary_prompt(chunk_text)
    if extra_instruction:
        prompt = f"{prompt}\n\nAdditional instruction:\n{extra_instruction.strip()}"

    response = await async_chat_wrapper(
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        model_size=model_name,
    )
    return response if isinstance(response, str) else str(response)


async def summarize_chunks_parallel(
    chunks: list[str],
    models: list[str] | None = None,
    max_concurrent: int = 5,
    temperature: float = 0.1,
    extra_instruction: str | None = None,
    verbose: bool = True,
) -> list[str]:
    """Summarize chunks in parallel with semaphore control and round-robin model selection."""
    if not chunks:
        return []

    active_models = models or DEFAULT_MODELS
    if not active_models:
        raise ValueError("models cannot be empty")

    semaphore = asyncio.Semaphore(max_concurrent)
    model_cycle = cycle(active_models)
    model_per_idx = [next(model_cycle) for _ in range(len(chunks))]

    async def _run(i: int, text: str) -> tuple[int, str]:
        model_name = model_per_idx[i]
        async with semaphore:
            if verbose:
                print(f"Summarizing chunk {i + 1}/{len(chunks)} with model={model_name}")
            summary = await summarize_single_chunk(
                chunk_text=text,
                model_name=model_name,
                temperature=temperature,
                extra_instruction=extra_instruction,
            )
            return i, summary

    tasks = [_run(i, text) for i, text in enumerate(chunks)]
    results = await asyncio.gather(*tasks)
    results.sort(key=lambda x: x[0])
    return [summary for _, summary in results]


def group_chunks(chunks: list[str], n_chunks: int) -> list[str]:
    """Group chunks in contiguous blocks of n_chunks and concatenate each block."""
    if n_chunks <= 0:
        raise ValueError("n_chunks must be >= 1")
    grouped: list[str] = []
    for i in range(0, len(chunks), n_chunks):
        grouped.append("\n\n".join(chunks[i:i + n_chunks]))
    return grouped


async def map_reduce_pipeline(
    chunks: list[str],
    n_chunks: int = 3,
    models: list[str] | None = None,
    max_concurrent: int = 5,
    temperature: float = 0.1,
    extra_instruction: str | None = None,
    verbose: bool = True,
) -> list[str]:
    """One map-reduce pass: group -> summarize groups -> return new groups."""
    grouped = group_chunks(chunks, n_chunks=n_chunks)
    if verbose:
        print(f"Grouped {len(chunks)} chunks into {len(grouped)} groups (n_chunks={n_chunks})")

    summaries = await summarize_chunks_parallel(
        grouped,
        models=models,
        max_concurrent=max_concurrent,
        temperature=temperature,
        extra_instruction=extra_instruction,
        verbose=verbose,
    )
    return summaries


async def run_recursive_map_reduce(
    initial_chunks: list[str],
    n_chunks: int = 3,
    target_max_chars: int = 20000,
    models: list[str] | None = None,
    max_concurrent: int = 5,
    temperature: float = 0.1,
    extra_instruction: str | None = None,
    max_rounds: int = 12,
    verbose: bool = True,
) -> dict[str, Any]:
    """Iteratively apply map-reduce until final summary length is below threshold."""
    if not initial_chunks:
        return {
            "rounds": [],
            "final_chunks": [],
            "final_summary": "",
            "final_length": 0,
        }

    rounds: list[dict[str, Any]] = []
    current_chunks = list(initial_chunks)

    for round_idx in range(1, max_rounds + 1):
        concatenated_before = "\n\n".join(current_chunks)
        before_len = len(concatenated_before)

        if verbose:
            print(f"\n=== Round {round_idx} ===")
            print(f"Input chunks: {len(current_chunks)} | input chars: {before_len}")

        next_chunks = await map_reduce_pipeline(
            chunks=current_chunks,
            n_chunks=n_chunks,
            models=models,
            max_concurrent=max_concurrent,
            temperature=temperature,
            extra_instruction=extra_instruction,
            verbose=verbose,
        )

        concatenated_after = "\n\n".join(next_chunks)
        after_len = len(concatenated_after)
        rounds.append(
            {
                "round": round_idx,
                "input_chunk_count": len(current_chunks),
                "output_chunk_count": len(next_chunks),
                "input_chars": before_len,
                "output_chars": after_len,
            }
        )

        if verbose:
            print(f"Output chunks: {len(next_chunks)} | output chars: {after_len}")

        current_chunks = next_chunks

        if after_len < target_max_chars:
            if verbose:
                print(f"Stopping: final summary length {after_len} < {target_max_chars}")
            break

        if len(current_chunks) == 1 and after_len >= before_len:
            if verbose:
                print("Stopping: only one chunk left and summary is not shrinking.")
            break

    final_summary = "\n\n".join(current_chunks)
    return {
        "rounds": rounds,
        "final_chunks": current_chunks,
        "final_summary": final_summary,
        "final_length": len(final_summary),
    }

Evaluation functions defined.
Visualization function defined.


In [2]:
pages = load_notion_pages("./rag_exp/notion_docs")
selected = select_pages(pages, keys=None)

if "block_reference" not in selected:
    raise KeyError("block_reference not found in rag_exp/notion_docs")

In [3]:
source_text = "\n\n".join(selected.values())
initial_chunks = chunk_with_recursive_chunker(source_text, chunk_size=20000)

print(f"Loaded pages: {len(pages)}")
print(f"Selected pages: {list(selected.keys())}")
print(f"Initial chunks (chunk_size={len(initial_chunks[0])}): {len(initial_chunks)}")

Loaded pages: 6
Selected pages: ['block_reference', 'create_block_endpoint', 'create_data_source_endpoint', 'create_page_endpoint', 'data_source_reference', 'page_reference']
Initial chunks (chunk_size=19434): 27


In [4]:
result = await run_recursive_map_reduce(
    initial_chunks=initial_chunks,
    n_chunks=3,
    target_max_chars=40000,
    models=["gemma27", "gemini-3.1-flash-lite-preview"],  # defaults to [gemma4, gemma12, gemma27]
    max_concurrent=5,
    temperature=0.1,
    extra_instruction=None,
    max_rounds=12,
    verbose=True,
)

print("\n=== Final ===")
print(f"Rounds executed: {len(result['rounds'])}")
print(f"Final length: {result['final_length']} chars")
print(f"Final chunk count: {len(result['final_chunks'])}")

2026-03-20 00:36:23 | WARNING  | src.all_functionality | Model size 'gemini-3.1-flash-lite-preview' not found in map, using as-is: gemini-3.1-flash-lite-preview
2026-03-20 00:36:23 | WARNING  | src.all_functionality | Model size 'gemini-3.1-flash-lite-preview' not found in map, using as-is: gemini-3.1-flash-lite-preview



=== Round 1 ===
Input chunks: 27 | input chars: 530952
Grouped 27 chunks into 9 groups (n_chunks=3)
Summarizing chunk 1/9 with model=gemma27
Summarizing chunk 2/9 with model=gemini-3.1-flash-lite-preview
Summarizing chunk 3/9 with model=gemma27
Summarizing chunk 4/9 with model=gemini-3.1-flash-lite-preview
Summarizing chunk 5/9 with model=gemma27


2026-03-20 00:36:25 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:36:25 | INFO     | openai._base_client | Retrying request to /chat/completions in 0.492395 seconds
2026-03-20 00:36:25 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:36:25 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:36:25 | INFO     | openai._base_client | Retrying request to /chat/completions in 0.498938 seconds
2026-03-20 00:36:25 | INFO     | openai._base_client | Retrying request to /chat/completions in 0.398786 seconds
2026-03-20 00:36:25 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Request

[async_chat_wrapper] Model: gemini-3.1-flash-lite-preview, finish_reason: stop
Summarizing chunk 6/9 with model=gemini-3.1-flash-lite-preview


2026-03-20 00:36:30 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:36:30 | INFO     | openai._base_client | Retrying request to /chat/completions in 3.852606 seconds
2026-03-20 00:36:30 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


...

[async_chat_wrapper] Model: gemini-3.1-flash-lite-preview, finish_reason: stop
Summarizing chunk 9/9 with model=gemma27


2026-03-20 00:36:40 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:36:40 | INFO     | openai._base_client | Retrying request to /chat/completions in 7.374159 seconds
2026-03-20 00:36:41 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:36:41 | INFO     | openai._base_client | Retrying request to /chat/completions in 7.805256 seconds
2026-03-20 00:36:42 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:36:42 | INFO     | openai._base_client | Retrying request to /chat/completions in 7.995317 seconds
2026-03-20 00:36:46 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Request

[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


2026-03-20 00:37:03 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:37:03 | INFO     | openai._base_client | Retrying request to /chat/completions in 7.693698 seconds
2026-03-20 00:37:04 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:37:04 | INFO     | openai._base_client | Retrying request to /chat/completions in 7.623140 seconds
2026-03-20 00:37:04 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-20 00:37:04 | INFO     | openai._base_client | Retrying request to /chat/completions in 6.429684 seconds
2026-03-20 00:37:07 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Request

CancelledError: 

In [22]:
print(result['final_summary'])

### Block Object Overview
A block object represents content within Notion.
*   **Common Fields:** `object` ("block"), `id` (UUIDv4), `parent` (object), `type` (enum), `created_time`, `created_by`, `last_edited_time`, `last_edited_by`, `has_children`, `in_trash`.
*   **Capabilities:** Fields marked with `*` are available to all integrations; others require read content capabilities.
*   **Children:** Use [Retrieve block children](/reference/get-block-children) to list blocks.

### Block Types
The `type` field determines the structure of the corresponding block type object. Supported types include:
`bookmark`, `breadcrumb`, `bulleted_list_item`, `callout`, `child_database`, `child_page`, `column`, `column_list`, `divider`, `embed`, `equation`, `file`, `heading_1`, `heading_2`, `heading_3`, `image`, `link_preview`, `numbered_list_item`, `paragraph`, `pdf`, `quote`, `synced_block`, `table`, `table_of_contents`, `table_row`, `template`, `to_do`, `toggle`, `transcription`, `unsupported`, `vi

- technically great
- may contain some hallucinations

In [23]:
print(await async_chat_wrapper(
    messages=[{"role": "user", "content": f"```{result['final_summary']}```How good is this description about Notion Block API?"}],
    temperature=0.1,
    model_size="gemini-3.1-flash-lite-preview",
))

2026-03-18 00:27:15 | WARNING  | src.all_functionality | Model size 'gemini-3.1-flash-lite-preview' not found in map, using as-is: gemini-3.1-flash-lite-preview


2026-03-18 00:27:18 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemini-3.1-flash-lite-preview, finish_reason: stop
The provided documentation is a highly accurate, comprehensive, and well-structured technical reference for the Notion API. It effectively synthesizes complex object hierarchies into a readable format suitable for developers.

### Strengths
*   **Logical Organization:** The separation of "Block Object Overview," "Block Type Details," and "Request Schemas" mirrors the actual API structure, making it easy to navigate.
*   **Constraint Clarity:** Explicitly calling out read-only constraints (e.g., `link_preview`, `meeting_notes`) and deprecated features (e.g., `template` creation) prevents common implementation errors.
*   **Technical Precision:** The inclusion of specific version requirements (`2026-03-11`), HTTP status codes, and data type constraints (e.g., `rich_text` limits, `table_width` requirements) provides the necessary detail for robust integration development.
*   **Schema Coverage:** The documentat

### Changelog

## Trials

'A block object represents content within Notion.

Example Block Object:

```json
{
	"object": "block",
	"id": "c02fc1d3-db8b-45c5-a222-27595b15aea7",
	"parent": {
		"type": "page_id",
		"page_id": "59833787-2cf9-4fdf-8782-e53db20768a5"
	},
	"created_time": "2022-03-01T19:05:00.000Z",
	"last_edited_time": "2022-07-06T19:41:00.000Z",
	"created_by": {
		"object": "user",
		"id": "ee5f0f84-409a-440f-983a-a5315961c6e4"
	},
	"last_edited_by": {
		"object": "user",
		"id": "ee5f0f84-409a-440f-983a-a5315961c6e4"
	},
	"has_children": false,
	"in_trash": false,
	"type": "heading_2",
	"heading_2": {
		"rich_text": [
			{
				"type": "text",
				"text": {
					"content": "Lacinato kale",
					"link": null
				},
				"annotations": {
					"bold": false,
					"italic": false,
					"strikethrough": false,
					"underline": false,
					"code": false,
					"color": "green"
				},
				"plain_text": "Lacinato kale",
				"href": null
			}
		],
		"color": "default",
      "is_toggleable": false
  	}
}
```

Key Fields:

| Field              | Type                                                                    |
| :----------------- | :---------------------------------------------------------------------- |
| `object`\*         | `string`                                                                | `"block"`
| `id`\*             | `string` (UUIDv4)                                                       |
| `parent`           | `object`                                                                |


Block Type Enumeration:

*   `"bookmark"`
*   `"breadcrumb"`
*   `"bulleted_list_item"`
*   `"callout"`
*   `"child_database"`
*   `"child_page"`
*   `"column"`
*   `"column_list"`
*   `"divider"`
*   `"embed"`
*   `"equation"`
*   `"file"`
*   `"heading_1"`
*   `"heading_2"`
*   `"heading_3"`
*   `"image"`
*   `"link_preview"`
*   `"numbered_list_item"`
*   `"paragraph"`
*   `"pdf"`
*   `"quote"`
*   `"synced_block"`
*   `"table"`
*   `"table_of_contents"`
*   `"table_row"`
*   `"template"`
*   `"to_do"`
*   `"toggle"`
*   `"transcription"`
*   `"unsupported"`
*   `"video"`

Date/Time Formats:

*   `created_time`: ISO 8601 date time (e.g., `"2020-03-17T19:10:04.968Z"`)
*   `last_edited_time`: ISO 8601 date time (e.g., `"2020-03-17T19:10:04.968Z"`)

User Objects:

*   `created_by`: Partial User object (e.g., `{"object": "user","id": "45ee8d13-687b-47ce-a5ca-6e2e45548c4b"}`)
*   `last_edited_by`: Partial User object (e.g., `{"object": "user","id": "45ee8d13-687b-47ce-a5ca-6e2e45548c4b"}`)

Boolean Fields:

*   `in_trash`: `boolean` (default: `false`)
*   `has_children`: `boolean` (default: `true`)

Block Properties:

*   `in_trash`: `boolean` (deprecated, alias for `in_trash`)
*   `has_children`: `boolean` (default: `true`)
*   `type`: String (block type)
*   `rich_text`: Array of rich text objects.
*   `color`: String (enum) - Values: `"blue"`, `"blue_background"`, `"brown"`, `"brown_background"`, `"default"`, `"gray"`, `"gray_background"`, `"green"`, `"green_background"`, `"orange"`, `"orange_background"`, `"yellow"`, `"pink"`, `"pink_background"`, `"purple"`, `"purple_background"`, `"red"`, `"red_background"`, `"yellow_background"`.

Block Types:

*   `callout`:  `rich_text`, `icon`, `color`
*   `child_database`: `title`
*   `child_page`: `title`
*   `heading_1`, `heading_2`, `heading_3`: `rich_text`, `color`, `is_toggleable`
*   `image`: File object
*   `link_preview`: `url` (read-only)
*   `paragraph`: `rich_text`
*   `quote`: `rich_text`, `color`
*   `synced_block`: `synced_from`, `children`
*   `table`: `table_width`, `has_column_header`, `has_row_header`
*   `table_row`: `cells`
*   `template`: (Deprecated)
*   `to_do`: `rich_text`, `checked`, `color`
*   `toggle`: `rich_text`, `color`
*   `video`: File object
'

### Block Object
A block represents content in Notion. Every block object contains:
*   `object`: Always `"block"`.
*   `id`: UUIDv4.
*   `type`: Enum (e.g., `"paragraph"`, `"heading_1"`, `"to_do"`, `"table"`).
*   `created_time`, `last_edited_time`: ISO 8601 strings.
*   `created_by`, `last_edited_by`: Partial User objects.
*   `has_children`: Boolean.
*   `in_trash`: Boolean.
*   `parent`: Parent object.
*   `{type}`: Object containing type-specific data.

**Supported Block Types:**
`bookmark`, `breadcrumb`, `bulleted_list_item`, `callout`, `child_database`, `child_page`, `column`, `column_list`, `divider`, `embed`, `equation`, `file`, `heading_1/2/3`, `image`, `link_preview`, `numbered_list_item`, `paragraph`, `pdf`, `quote`, `synced_block`, `table`, `table_of_contents`, `table_row`, `template`, `to_do`, `toggle`, `meeting_notes` (formerly `transcription`), `unsupported`, `video`.

**Child-Supporting Blocks:**
`bulleted_list_item`, `callout`, `child_database`, `child_page`, `column`, `heading_1/2/3` (if `is_toggleable: true`), `meeting_notes`, `numbered_list_item`, `paragraph`, `quote`, `synced_block`, `table`, `template`, `to_do`, `toggle`.

---

### Append Block Children
**Endpoint:** `PATCH /v1/blocks/{block_id}/children`

*   **Function:** Appends new children to a parent block.
*   **Limits:** Max 100 children per request; max 2 levels of nesting.
*   **Positioning:** Use the `position` parameter:
    *   `{ "type": "end" }` (Default)
    *   `{ "type": "start" }`
    *   `{ "type": "after_block", "after_block": { "id": "<block_id>" } }`

**Request Body:**
```json
{
  "children": [ /* Array of block objects */ ],
  "position": { "type": "..." }
}
```

**Key Constraints:**
*   Requires "insert content" capabilities.
*   `table_width` is immutable after creation.
*   `link_preview` and `meeting_notes` are read-only.
*   `synced_block` content cannot be updated via API.
*   `unsupported` blocks contain a `block_type` string (e.g., `"form"`, `"button"`).

### Create a Data Source
**Endpoint:** `POST /v1/data_sources`
**Summary:** Adds a data source to an existing database.

#### Request Body
| Field | Type | Description |
| :--- | :--- | :--- |
| `parent` | `object` | Specifies the parent database. Required: `database_id` (string). |
| `properties` | `object` | Property schema of the data source. |
| `title` | `array` | Title of the data source (max 100 items). |
| `icon` | `object` | Page icon (optional). |

**`parent` Schema (`parentOfDataSourceRequest`):**
*   `type`: `database_id` (const)
*   `database_id`: `idRequest` (string)

#### Property Configuration
The `properties` object uses `propertyConfigurationRequest`, which supports the following types:
*   `number`, `formula`, `select`, `multi_select`, `status`, `relation`, `rollup`, `unique_id`, `title`, `rich_text`, `url`, `people`, `files`, `email`, `phone_number`, `date`, `checkbox`, `created_by`, `created_time`, `last_edited_by`, `last_edited_time`, `button`, `location`, `verification`, `last_visited_time`, `place`.

#### Responses
*   **200 OK:** Returns `partialDataSourceObjectResponse` or `dataSourceObjectResponse`.
*   **Error Codes:** 400 (`invalid_json`, `invalid_request_url`, `invalid_request`, `missing_version`, `validation_error`), 401 (`unauthorized`), 403 (`restricted_resource`), 404 (`object_not_found`), 409 (`conflict_error`), 429 (`rate_limited`), 500 (`internal_server_error`), 503 (`service_unavailable`).

#### Headers
*   `Notion-Version`: Must be `2026-03-11`.
*   `Authorization`: Bearer token required.

#### Key Constraints
*   Managing database views is not supported via this API.
*   `properties` follow the same structure as the `initial_data_source[properties]` schema used in the "Create a database" API.

### Create a Page (`POST /v1/pages`)

**Endpoint:** `POST /v1/pages`
**Header:** `Notion-Version: 2026-03-11` (Required)

#### Request Body Parameters
*   **`parent`**: Required (unless creating a workspace-level page for public integrations).
    *   `page_id`: `{ "page_id": "uuid", "type": "page_id" }`
    *   `database_id`: `{ "database_id": "uuid", "type": "database_id" }`
    *   `data_source_id`: `{ "data_source_id": "uuid", "type": "data_source_id" }`
    *   `workspace`: `{ "workspace": true, "type": "workspace" }`
*   **`properties`**: Object containing page properties.
    *   If child of a page: Only `title` is valid.
    *   If child of a data source: Keys must match the data source's property schema.
    *   **Unsupported properties**: `rollup`, `created_by`, `created_time`, `last_edited_by`, `last_edited_time`.
*   **`children`**: Array of block objects (max 100). Mutually exclusive with `markdown`.
*   **`markdown`**: String of Notion-flavored Markdown. Mutually exclusive with `content`/`children`.
*   **`template`**:
    *   `type: "none"` (default)
    *   `type: "default"` (requires data source default template)
    *   `type: "template_id"`, `template_id: "uuid"`
    *   Optional `timezone`: IANA string (e.g., `America/New_York`).
*   **`icon`**: `pageIconRequest` (file, emoji, external, or custom_emoji).
*   **`cover`**: `pageCoverRequest` (file or external).
*   **`position`**: `pagePositionSchema` (`after_block`, `page_start`, or `page_end`).

#### Constraints & Behavior
*   **Capabilities**: Integration must have "Insert Content" capabilities on the target parent.
*   **Templates**: When `template` is used, `children` is not allowed. The page is returned blank, and the template is applied asynchronously.
*   **Markdown**: Newlines must be encoded as `\n`.
*   **Errors**: Returns standard HTTP status codes (400, 401, 403, 404, 409, 429, 500, 503).

#### Response
Returns a `pageObjectResponse` or `partialPageObjectResponse`.

### Data Source Object
A **Data Source** represents a table within a Notion database. Pages are children of data sources.

**Fields:**
* `object`: Always `"data_source"`.
* `id`: UUID string.
* `properties`: Schema of properties (key: property name, value: Property object).
* `parent`: Parent database information.
* `database_parent`: The database's parent (grandparent of the data source).
* `created_time` / `last_edited_time`: ISO 8601 date-time strings.
* `created_by` / `last_edited_by`: Partial User objects.
* `title` / `description`: Array of rich text objects.
* `icon`: File or Emoji object.
* `in_trash`: Boolean (replaces deprecated `archived`).

**Constraints:**
* Recommended maximum schema size: **50KB**.
* API version `2025-09-03` introduced specific management APIs for data sources.

---

### Page Object
A **Page** contains property values and content (blocks).

**Fields:**
* `object`: Always `"page"`.
* `id`: UUIDv4 string.
* `created_time` / `last_edited_time`: ISO 8601 date-time strings.
* `created_by` / `last_edited_by`: Partial User objects.
* `in_trash`: Boolean (replaces deprecated `archived`).
* `icon` / `cover`: File or Emoji objects.
* `properties`: Property values. If `parent.type` is `data_source_id`, values conform to the data source schema; otherwise, only `title` is valid.
* `parent`: Parent object.
* `url`: Notion page URL.
* `public_url`: Published web URL or `null`.

---

### Block Types (Request Schemas)
The following block types are defined as objects with `type` and `object` (always `"block"`) constants:

* **Paragraph**: `paragraph` (contentWithRichTextAndColorRequest)
* **Bulleted List Item**: `bulleted_list_item` (contentWithRichTextAndColorRequest)
* **Numbered List Item**: `numbered_list_item` (contentWithRichTextAndColorRequest)
* **Quote**: `quote` (contentWithRichTextAndColorRequest)
* **To Do**: `to_do` (rich_text, checked, color)
* **Toggle**: `toggle` (contentWithRichTextAndColorRequest)
* **Template**: `template` (contentWithRichTextRequest)
* **Callout**: `callout` (rich_text, icon, color)
* **Synced Block**: `synced_block` (synced_from: {block_id})

---

### Property Value Responses
Property values are categorized as `simplePropertyValueResponse` or `arrayBasedPropertyValueResponse`.

* **Simple Types**: `number`, `url`, `select`, `multi_select`, `status`, `date`, `email`, `phone_number`, `checkbox`, `files`, `created_by`, `created_time`, `last_edited_by`, `last_edited_time`, `formula`, `button`, `unique_id`, `verification`, `place`.
* **Array-based Types**: `title`, `rich_text`, `people`, `relation`.

**Rollup Property:**
* `type`: Always `rollup`.
* `rollup`: Contains `function` (e.g., `count`, `sum`, `average`, `median`, `min`, `max`, `percent_not_empty`) and specific value types (`number`, `date`, or `array`).

---

### Rich Text and Mentions
* **Rich Text Items**: `text`, `mention`, `equation`.
* **Annotations**: `bold`, `italic`, `strikethrough`, `underline`, `code`, `color`.
* **Mentions**: Support `user`, `date`, `link_preview`, `link_mention`, `page`, `database`, `template_mention`, and `custom_emoji`.

### Block Object Overview
A block object represents content within Notion.
*   **Common Fields:** `object` ("block"), `id` (UUIDv4), `parent` (object), `type` (enum), `created_time`, `created_by`, `last_edited_time`, `last_edited_by`, `has_children`, `in_trash`.
*   **Capabilities:** Fields marked with `*` are available to all integrations; others require read content capabilities.
*   **Children:** Use [Retrieve block children](/reference/get-block-children) to list blocks.

### Block Types
The `type` field determines the structure of the corresponding block type object. Supported types include:
`bookmark`, `breadcrumb`, `bulleted_list_item`, `callout`, `child_database`, `child_page`, `column`, `column_list`, `divider`, `embed`, `equation`, `file`, `heading_1`, `heading_2`, `heading_3`, `image`, `link_preview`, `numbered_list_item`, `paragraph`, `pdf`, `quote`, `synced_block`, `table`, `table_of_contents`, `table_row`, `template`, `to_do`, `toggle`, `transcription`, `unsupported`, `video`.

### Block Type Details
*   **Audio:** Contains a `file` object (type: `external` or `file_upload`). Supported: `.mp3`, `.wav`, `.ogg`, `.oga`, `.m4a`.
*   **Bookmark:** Contains `caption` (rich text array) and `url` (string).
*   **Breadcrumb:** Empty object.
*   **Bulleted List Item:** Contains `rich_text`, `color` (enum), and `children` (array of block objects).
*   **Callout:** Contains `rich_text`, `icon` (emoji or file object), and `color`.
*   **Child Database:** Contains `title` (string).
*   **Child Page:** Contains `title` (string).
*   **Code:** Contains `caption`, `rich_text`, and `language` (enum).
*   **Column List:** Parent for columns.
*   **Column:** Contains `width_ratio` (number 0–1). Must be appended to `column_list`.
*   **Divider:** Empty object.
*   **Embed:** Contains `url` (string).
*   **Equation:** Contains `expression` (KaTeX string).
*   **File:** Contains `caption`, `type` (`file`, `external`, `file_upload`), `name`, and specific file objects (`file`, `external`, or `file_upload`).
*   **Headings (1, 2, 3):** Contain `rich_text` and `color`.

### Child Block Support
The following types support nested children:
`bulleted_list_item`, `callout`, `child_database`, `child_page`, `column`, `heading_1/2/3` (if `is_toggleable` is `true`), `meeting_notes`, `numbered_list_item`, `paragraph`, `quote`, `synced_block`, `table`, `template`, `to_do`, `toggle`.

### Notes
*   **Unsupported Types:** Appear with `type: "unsupported"` and include a `block_type` field.
*   **Rich Text:** Many blocks include a `rich_text` object with a `plain_text` property.
*   **Colors:** Most blocks supporting color use the same enum: `"blue"`, `"blue_background"`, `"brown"`, `"brown_background"`, `"default"`, `"gray"`, `"gray_background"`, `"green"`, `"green_background"`, `"orange"`, `"orange_background"`, `"yellow"`, `"pink"`, `"pink_background"`, `"purple"`, `"purple_background"`, `"red"`, `"red_background"`, `"yellow_background"`.

### Block Types Summary

#### Heading Blocks
*   **Types:** `heading_1`, `heading_2`, `heading_3`.
*   **Properties:** `rich_text` (array), `color` (enum), `is_toggleable` (boolean).
*   **`is_toggleable`:** If `true`, the block toggles and supports children; if `false`, it is static.

#### Image
*   **Properties:** `image` (file object).
*   **Supported Types:** `.bmp`, `.gif`, `.heic`, `.jpeg`, `.jpg`, `.png`, `.svg`, `.tif`, `.tiff`.
*   **Constraint:** Must be directly hosted; `url` cannot point to a retrieval service.

#### Link Preview
*   **Properties:** `url` (string).
*   **Constraint:** Read-only; cannot be created or appended via API.

#### Meeting Notes
*   **Properties:** `title` (rich text array), `status` (enum: `"transcription_not_started"`, `"transcription_paused"`, `"transcription_in_progress"`, `"summary_in_progress"`, `"notes_ready"`), `children` (object with `summary_block_id`, `notes_block_id`, `transcript_block_id`), `calendar_event` (object), `recording` (object).
*   **Constraint:** Read-only. Renamed from `transcription` in API version `2026-03-11`.

#### Mention
*   **Context:** Child of a rich text object nested in a paragraph.
*   **Types:** `"database"`, `"date"`, `"link_preview"`, `"page"`, `"user"`.

#### Numbered List Item
*   **Properties:** `rich_text`, `color` (enum), `list_start_index` (integer, optional), `list_format` (enum: `"numbers"`, `"letters"`, `"roman"`, optional), `children`.

#### Paragraph
*   **Properties:** `rich_text`, `color` (enum), `children`.

#### PDF
*   **Properties:** `caption` (rich text array), `type` (`"file"`, `"external"`, `"file_upload"`), `external`/`file`/`file_upload` (file object).
*   **Constraint:** Only supports `.pdf`.

#### Quote
*   **Properties:** `rich_text`, `color` (enum), `children`.

#### Synced Block
*   **Original:** `synced_from` is `null`; contains `children`.
*   **Duplicate:** `synced_from` contains `block_id` (UUIDv4).
*   **Constraint:** Content cannot be updated via API.

#### Table
*   **Properties:** `table_width` (integer, immutable after creation), `has_column_header` (boolean), `has_row_header` (boolean).
*   **Table Row:** Contains `cells` (array of array of rich text objects).
*   **Constraint:** When creating, at least one `table_row` must have a `cells` array length equal to `table_width`.

#### Table of Contents
*   **Properties:** `color` (enum).

#### Template
*   **Properties:** `rich_text`, `children`.
*   **Constraint:** Creation no longer supported as of March 27, 2023.

#### To Do
*   **Properties:** `rich_text`, `checked` (boolean), `color` (enum), `children`.

#### Toggle
*   **Properties:** `rich_text`, `color` (enum), `children`.

#### Unsupported
*   **Properties:** `block_type` (string).
*   **Note:** Informational only; does not expose content.

#### Video
*   **Properties:** `video` (file object).
*   **Supported Types:** `.amv`, `.asf`, `.avi`, `.f4v`, `.flv`, `.gifv`, `.mkv`, `.mov`, `.mpg`, `.mpeg`, `.mpv`, `.mp4`, `.m4v`, `.qt`, `.wmv`, and YouTube (`embed` or `watch` links).

---

### Append Block Children Endpoint
*   **Endpoint:** `PATCH /v1/blocks/{block_id}/children`
*   **Limits:** Max 100 blocks per request; max 2 levels of nesting.
*   **Positioning:** Use `position` parameter:
    *   `{ "type": "end" }` (default)
    *   `{ "type": "start" }`
    *   `{ "type": "after_block", "after_block": { "id": "<block_id>" } }`
*   **Constraint:** Requires "insert content" capability. Existing blocks cannot be moved.

### Block Request Schemas
All block objects require `object: "block"` and a specific `type` constant.

*   **Table of Contents**: `table_of_contents` (requires `color`).
*   **Link To Page**: `link_to_page` (requires `page_id`, `database_id`, or `comment_id`).
*   **Table Row**: `table_row` (requires `cells` array of `richTextItemRequest`).
*   **Table**: `table` (requires `table_width` (int >= 1) and `children` array of `tableRowRequest`).
*   **Column List**: `column_list` (requires `children` array of `columnBlockWithChildrenRequest`, min 2).
*   **Column**: `column` (requires `children` array, optional `width_ratio` 0–1).
*   **Headings (1, 2, 3)**: `heading_1`, `heading_2`, `heading_3` (require `rich_text` array, optional `color`, `is_toggleable`, `children`).
*   **Paragraph, Bulleted/Numbered List, Quote, Toggle**: Require `rich_text` array, optional `color`, `children`.
*   **To Do**: `to_do` (requires `rich_text`, optional `color`, `children`, `checked`).
*   **Template**: `template` (requires `rich_text`, optional `children`).
*   **Callout**: `callout` (requires `rich_text`, optional `color`, `children`, `icon`).
*   **Synced Block**: `synced_block` (requires `synced_from` with `block_id`, optional `children`).

**Constraints**: `rich_text` and `children` arrays are limited to `maxItems: 100`.

### Content Position Schema
Used for positioning:
*   `after_block`: Requires `id`.
*   `start`: No additional properties.
*   `end`: No additional properties.

### API Error Codes
All errors return `publicApiCommonErrorResponse` and a `status` code:
*   **400**: `invalid_json`, `invalid_request_url`, `invalid_request`, `missing_version`, `validation_error`.
*   **401**: `unauthorized`.
*   **403**: `restricted_resource`.
*   **404**: `object_not_found`.
*   **409**: `conflict_error`.
*   **429**: `rate_limited`.
*   **500**: `internal_server_error`.
*   **503**: `service_unavailable`.

### Media & Specialized Content
*   **Media (Url/Caption)**: `url` (string), `caption` (rich text).
*   **Media (File/Caption)**: `external` or `file_upload` types, `caption` (rich text).
*   **Code**: Requires `rich_text`, `language` (enum list provided), optional `caption`.
*   **Equation**: Requires `expression` (string).
*   **API Color Enum**: `default`, `gray`, `brown`, `orange`, `yellow`, `green`, `blue`, `purple`, `pink`, `red`, plus `_background` variants.

### Block Object Responses
Standard response fields for all block objects:
*   `object`: "block"
*   `id`: UUID
*   `type`: String
*   `parent`: `parentForBlockBasedObjectResponse`
*   `created_time`, `last_edited_time`: ISO 8601 date-time
*   `created_by`, `last_edited_by`: `partialUserObjectResponse`
*   `has_children`, `in_trash`: Boolean

### Block Object Responses
All block objects share these required fields: `type`, `parent`, `object` (literal: `block`), `id` (uuid), `created_time` (date-time), `created_by`, `last_edited_time` (date-time), `last_edited_by`, `has_children` (boolean), and `in_trash` (boolean).

*   **Equation:** `equation` (expressionObjectResponse).
*   **Code:** `code` (rich_text, caption, language).
*   **Callout:** `callout` (rich_text, color, icon).
*   **Table of Contents:** `table_of_contents` (color).
*   **Link To Page:** `link_to_page` (one of: `page_id`, `database_id`, or `comment_id`).
*   **Table:** `table` (contentWithTableResponse).
*   **Table Row:** `table_row` (contentWithTableRowResponse).
*   **Meeting Notes:** `meeting_notes` (transcriptionBlockResponse).
*   **Embed/Bookmark:** `embed`/`bookmark` (mediaContentWithUrlAndCaptionResponse).
*   **Image/Video/Pdf/Audio:** `image`/`video`/`pdf`/`audio` (mediaContentWithFileAndCaptionResponse).
*   **File:** `file` (mediaContentWithFileNameAndCaptionResponse).
*   **Link Preview:** `link_preview` (mediaContentWithUrlResponse).
*   **Unsupported:** `unsupported` (block_type).
*   **Divider/Breadcrumb/Column List:** Empty objects.

### Request Schemas
*   **Rich Text Items:**
    *   `text`: `content` (max 2000 chars), optional `link` (url).
    *   `mention`: `user`, `date`, `page`, `database`, `template_mention`, or `custom_emoji`.
    *   `equation`: `expression` (KaTeX string).
*   **Block Creation (`blockObjectRequestWithoutChildren`):**
    *   Supports specific types: `embed`, `bookmark`, `image`, `video`, `pdf`, `file`, `audio`, `code`, `equation`, `divider`, `breadcrumb`, `table_of_contents`, `link_to_page`, `table_row`, `heading_1/2/3`, `paragraph`, `bulleted_list_item`, `numbered_list_item`, `quote`, `to_do`, `toggle`, `template`, `callout`, `synced_block`.
*   **Page Icons:**
    *   `file_upload`: `id` (status `uploaded`).
    *   `emoji`: `emoji` (string).
    *   `external`: `url`.
    *   `custom_emoji`: `id`, `name`, `url`.

### Common Components
*   **Parent Types:** `database_id`, `data_source_id`, `page_id`, `block_id`, `workspace`.
*   **Annotations:** `bold`, `italic`, `strikethrough`, `underline`, `code`, `color`.
*   **Error Response:** `object` (literal: `error`), `message`, `additional_data`.
*   **Constraints:**
    *   `rich_text` arrays: max 100 items.
    *   `stringRequest`: 1–100 characters.
    *   `textRequest`: 1–2000 characters.
    *   `table_width`: min 1.

### Create a Data Source
**Endpoint:** `POST /v1/data_sources`  
**Description:** Adds a data source to an existing database. A standard "table" view is created automatically.

#### Request Body
| Field | Type | Required | Description |
| :--- | :--- | :--- | :--- |
| `parent` | `object` | Yes | Specifies the parent database via `database_id`. |
| `properties` | `object` | Yes | Property schema; uses `propertyConfigurationRequest` structure. |
| `title` | `array` | No | Title of the data source (max 100 items). |
| `icon` | `object` | No | Page icon (`pageIconRequest` or `null`). |

**`parent` Object:**
* `type`: `const: database_id`
* `database_id`: `idRequest` (ID of the parent database)

#### Response
* **Success (200):** Returns `partialDataSourceObjectResponse` or `dataSourceObjectResponse`.
* **Error Codes:** `400` (invalid_json, invalid_request, validation_error, etc.), `401` (unauthorized), `403` (restricted_resource), `404` (object_not_found), `409` (conflict_error), `429` (rate_limited), `500` (internal_server_error), `503` (service_unavailable).

#### Headers
* `Notion-Version`: Must be `2026-03-11`.
* `Authorization`: Bearer token required.

#### Property Configuration Types
The `properties` field supports the following configurations:
* `number`, `formula`, `select`, `multi_select`, `status`, `relation`, `rollup`, `unique_id`, `title`, `rich_text`, `url`, `people`, `files`, `email`, `phone_number`, `date`, `checkbox`, `created_by`, `created_time`, `last_edited_by`, `last_edited_time`, `button`, `location`, `verification`, `last_visited_time`, `place`.

#### Rich Text Items
Requests for `title` or `description` fields use `richTextItemRequest`, which supports:
* `text`: Requires `content` (max 2000 chars).
* `mention`: Supports `user`, `date`, `page`, `database`, `template_mention`, or `custom_emoji`.
* `equation`: Requires `expression` (KaTeX compatible).

### Endpoint: `POST /v1/pages`
**Summary:** Creates a new page as a child of an existing page, data source, or at the workspace level.

#### Request Body Parameters
*   **`parent`** (Optional): Defines the parent container.
    *   `page_id`: `{ "page_id": "string", "type": "page_id" }`
    *   `database_id`: `{ "database_id": "string", "type": "database_id" }`
    *   `data_source_id`: `{ "data_source_id": "string", "type": "data_source_id" }`
    *   `workspace`: `{ "workspace": true, "type": "workspace" }` (Public integrations only).
*   **`properties`**: Object containing page properties. Keys must match the parent data source's properties.
    *   Unsupported properties: `rollup`, `created_by`, `created_time`, `last_edited_by`, `last_edited_time`.
*   **`children`**: Array of block objects to populate the page. (Not allowed if `template` is provided).
*   **`template`**: Specifies a template to apply.
    *   `type`: `none` (default), `default`, or `template_id`.
    *   `template_id`: The ID of the template page.
    *   `timezone`: IANA timezone string (e.g., `America/New_York`).

#### Requirements & Constraints
*   **Capabilities**: Integration must have "Insert Content" capabilities on the target parent.
*   **Authentication**: Requires `bearerAuth` (Bearer token).
*   **Error Codes**: 403 (Forbidden) if missing "Insert Content" capabilities; `validation_error` for invalid timezones.
*   **Markdown**: Newlines in `markdown` parameters must be encoded as `\n`.
*   **Asynchronous Processing**: When applying a template, the page is returned blank initially; template application occurs asynchronously.

#### Response
*   Returns a new [page object](/reference/page).

### API Specification Summary

#### Request Header
*   **`Notion-Version`**: Required. Must be `'2026-03-11'`.

#### Page Properties (Schema Types)
*   **`select`**: Requires `select` object.
*   **`multi_select`**: Requires `multi_select` array (max 100 items).
*   **`people`**: Requires `people` array (max 100 items) containing `partialUserObjectRequest` or `groupObjectRequest`.
*   **`email`**: Requires `email` (string or null).
*   **`phone_number`**: Requires `phone_number` (string or null).
*   **`date`**: Requires `date` object (start date, optional end date, optional time_zone).
*   **`checkbox`**: Requires `checkbox` (boolean).
*   **`relation`**: Requires `relation` array (max 100 items).
*   **`files`**: Requires `files` array (max 100 items) of `internalOrExternalFileWithNameRequest` or `fileUploadWithOptionalNameRequest`.
*   **`status`**: Requires `status` object (id, name, color, description).
*   **`place`**: Requires `place` object (lat, lon, optional name, address, aws_place_id, google_place_id).

#### Page Configuration
*   **`icon`**: `pageIconRequest` (file_upload, emoji, external, or custom_emoji).
*   **`cover`**: `pageCoverRequest` (file_upload or external).
*   **`content` / `children`**: Array of `blockObjectRequest` (max 100 items).
*   **`markdown`**: String (mutually exclusive with `content`/`children`).
*   **`template`**:
    *   `type: none`
    *   `type: default` (with optional `timezone`)
    *   `type: template_id` (with `template_id` and optional `timezone`)
*   **`position`**: `pagePositionSchema` (`after_block`, `page_start`, or `page_end`).

#### Block Types (`blockObjectRequest`)
Supported types include: `embed`, `bookmark`, `image`, `video`, `pdf`, `file`, `audio`, `code`, `equation`, `divider`, `breadcrumb`, `table_of_contents`, `link_to_page`, `table_row`, `table`, `column_list`, `column`, `heading_1`, `heading_2`, `heading_3`, `paragraph`, `bulleted_list_item`, `numbered_list_item`, `quote`, `to_do`, `toggle`, `template`, `callout`, and `synced_block`.

#### Common Data Constraints
*   **`textRequest`**: 1–2000 characters.
*   **`stringRequest`**: 1–100 characters.
*   **`selectColor`**: `default`, `gray`, `brown`, `orange`, `yellow`, `green`, `blue`, `purple`, `pink`, `red`.
*   **`richTextItemRequest`**: Supports `text`, `mention`, and `equation` types.
*   **`templateTimezone`**: IANA timezone string (e.g., `America/New_York`).

#### Error Responses
*   **400**: `invalid_json`, `invalid_request_url`, `invalid_request`, `missing_version`, `validation_error`.
*   **401**: `unauthorized`.
*   **403**: `restricted_resource`.
*   **404**: `object_not_found`.
*   **409**: `conflict_error`.
*   **429**: `rate_limited`.
*   **500**: `internal_server_error`.
*   **503**: `service_unavailable`.

### Block Objects
Blocks are defined by a `type` and an `object` (always `block`). Supported types include:
*   **Media:** `embed`, `bookmark`, `image`, `video`, `pdf`, `file`, `audio`.
*   **Content:** `paragraph`, `heading_1`, `heading_2`, `heading_3`, `bulleted_list_item`, `numbered_list_item`, `quote`, `to_do`, `toggle`, `template`, `callout`, `synced_block`.
*   **Structural/Other:** `code`, `equation`, `divider`, `breadcrumb`, `table_of_contents`, `link_to_page`, `table`, `table_row`, `column_list`, `column`.

### Key Request Schemas
*   **`contentWithExpressionRequest`**: Requires `expression` (string).
*   **`tableRequestWithTableRowChildren`**: Requires `table_width` (integer, min 1) and `children` (array of `tableRowRequest`, 1–100 items).
*   **`columnListRequest`**: Requires `children` (array of `columnBlockWithChildrenRequest`, 2–100 items).
*   **`columnWithChildrenRequest`**: Requires `children` (array of `blockObjectWithSingleLevelOfChildrenRequest`, max 100). Optional `width_ratio` (number, 0 < x < 1).
*   **`annotationRequest`**: Boolean flags for `bold`, `italic`, `strikethrough`, `underline`, `code`; `color` (enum `apiColor`).

### API Colors (`apiColor`)
`default`, `gray`, `brown`, `orange`, `yellow`, `green`, `blue`, `purple`, `pink`, `red`, and their `_background` variants.

### Page Property Value Responses
*   **Simple:** `number`, `url`, `select`, `multi_select`, `status`, `date`, `email`, `phone_number`, `checkbox`, `files`, `created_by`, `created_time`, `last_edited_by`, `last_edited_time`, `formula`, `button`, `unique_id`, `verification`, `place`.
*   **Array-based:** `title`, `rich_text`, `people`, `relation`.
*   **Rollup:** `partialRollupPropertyResponse` (type `rollup`).

### Parent Types
`databaseParentResponse` (`database_id`), `dataSourceParentResponse` (`data_source_id`, `database_id`), `pageIdParentForBlockBasedObjectResponse` (`page_id`), `blockIdParentForBlockBasedObjectResponse` (`block_id`), `workspaceParentForBlockBasedObjectResponse` (`workspace`: true).

### Page Icons/Covers
*   **Icons:** `emoji`, `file`, `external`, `custom_emoji`.
*   **Covers:** `file`, `external`.

### Error Response
`publicApiCommonErrorResponse` contains `object` (const `error`), `message` (string), and optional `additional_data` (object).

### Data Source Object
A data source represents a table within a Notion database.

**Fields:**
* `object`: Always `"data_source"`.
* `id`: UUID string.
* `properties`: Schema object (key: property name, value: Property object).
* `parent`: Parent database information.
* `database_parent`: Grandparent database information.
* `created_time` / `last_edited_time`: ISO 8601 strings.
* `created_by` / `last_edited_by`: Partial User objects.
* `title`: Array of rich text objects.
* `description`: Array of rich text objects.
* `icon`: File or Emoji object.
* `in_trash`: Boolean (replaces deprecated `archived`).

**Constraints:** Recommended maximum schema size is 50KB.

---

### Page Object
A Page contains property values conforming to its parent's schema.

**Fields:**
* `object`: Always `"page"`.
* `id`: UUIDv4 string.
* `created_time` / `last_edited_time`: ISO 8601 strings.
* `created_by` / `last_edited_by`: Partial User objects.
* `in_trash`: Boolean (replaces deprecated `archived`).
* `icon` / `cover`: File or Emoji objects.
* `parent`: Parent object.
* `properties`: Property values. If `parent.type` is `data_source_id`, values conform to the data source schema.
* `url`: Notion page URL.
* `public_url`: Published web URL or `null`.

---

### Property Value Schemas
* **Unique ID:** `prefix` (string/null), `number` (number/null).
* **Place:** `lat` (number), `lon` (number), `name` (string/null), `address` (string/null), `aws_place_id` (string/null), `google_place_id` (string/null).
* **Formula:** `type` (boolean, date, number, or string) with corresponding value field.
* **Verification:** `state` (`unverified` or `verified`/`expired`), `date`, `verified_by`.
* **Rich Text:** `plain_text`, `href`, `annotations` (bold, italic, strikethrough, underline, code, color). Types: `text`, `mention`, `equation`.
* **Mention:** Types include `user`, `date`, `link_preview`, `link_mention`, `page`, `database`, `template_mention`, `custom_emoji`.
* **File/External:** `type` (`file` or `external`), `name`, and specific object (`file` URL or `external` URL).
* **User:** `id`, `object` (`user`), `name`, `avatar_url`. Types: `person` (email) or `bot` (owner, workspace_id, workspace_limits, workspace_name).
* **Group:** `id`, `object` (`group`), `name`.
* **Rollup Function Enum:** `count`, `count_values`, `empty`, `not_empty`, `unique`, `show_unique`, `percent_empty`, `percent_not_empty`, `sum`, `average`, `median`, `min`, `max`, `range`, `earliest_date`, `latest_date`, `date_range`, `checked`, `unchecked`, `percent_checked`, `percent_unchecked`, `count_per_group`, `percent_per_group`, `show_original`.